# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields with @id references
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'No name')}\n  Description: {rs.get('description', 'No description')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields (@id):")
    for field in fields:
        # field can be {'@id': '...'} or a simple @id
        if isinstance(field, dict):
            print(f"    - {field['@id']}")
        else:
            print(f"    - {field}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect list of available record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Load data from each available record set (if any)
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading data from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"  Could not load records: {e}")
print(f"\nRecord set IDs loaded: {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA for the first record set (if available)
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if len(dataframes) > 0:
    selected_rs_id = list(dataframes.keys())[0]
    df = dataframes[selected_rs_id]
    print(f"Selected record set for EDA: {selected_rs_id}")
    
    # Show list of fields/columns
    print(f"Fields (columns): {df.columns.tolist()}")
    # Try to select a likely numeric field, e.g. using 'coverage', 'peptide', 'MW', or first numeric column
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to coerce columns to numeric if not already
        for col in df.columns:
            df[col + "_numeric"] = pd.to_numeric(df[col], errors='coerce')
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].notna().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head(3))
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head(3))
        
        # Try group by another field (use first object/categorical)
        obj_cols = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
        if obj_cols:
            group_field = obj_cols[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head(3))
        else:
            print("No groupable field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if len(dataframes) > 0 and numeric_cols:
    # Histogram of the selected numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in {selected_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    # If grouping, show boxplot if possible
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.